# 🛰️ HỆ THỐNG TRUYỀN DỮ LIỆU QUA VỆ TINH IRIDIUM - ROCKBLOCK 9603
## 📥 MODULE NHẬN (RECEIVER) - SERIAL / ID: 0235708

### 📖 Giới thiệu & Cơ chế nhận tin nhắn SBD (MT - Mobile Terminated):
* **Cơ chế P2P giữa 2 RockBLOCK trên mạng Iridium:**
  1. RockBLOCK **0235707** (Sender) gửi tin nhắn MO lên Gateway Iridium.
  2. Gateway Iridium / Rock Seven Core định tuyến tin nhắn tới IMEI của RockBLOCK **0235708** (Receiver) dưới dạng tin nhắn **MT**.
  3. Tin nhắn MT nằm trong hàng đợi tại trạm mặt đất Iridium Gateway chờ modem nhận kéo về.
  4. Modem 0235708 gọi lệnh `AT+SBDIX` (khởi tạo phiên SBD) để kiểm tra hộp thư (Mailbox check).
  5. Nếu `MT Status == 1`, tin nhắn đã được tải từ Gateway về bộ đệm MT của modem!
  6. Modem đọc nội dung bằng lệnh `AT+SBDRT` (Text) hoặc `AT+SBDRB` (Binary).
  7. Xóa bộ đệm MT bằng lệnh `AT+SBDD1` để chuẩn bị cho tin tiếp theo.


In [ ]:
# CELL 1: KHAI BÁO THƯ VIỆN VÀ CẤU HÌNH THIẾT BỊ
import time
import serial
import serial.tools.list_ports
from datetime import datetime

# ==============================================================================
# DANH SÁCH CỔNG SERIAL HIỆN TẠI TRÊN MÁY TÍNH
# ==============================================================================
print("🔍 Đang quét các cổng COM khả dụng trên máy:")
ports = list(serial.tools.list_ports.comports())
if not ports:
    print("  [!] Không tìm thấy cổng COM nào. Hãy kiểm tra kết nối cáp USB-FTDI!")
else:
    for p in ports:
        print(f"  -> Cổng: {p.device:<8} | Mô tả: {p.description}")

# ==============================================================================
# CẤU HÌNH KẾT NỐI CHO THIẾT BỊ NHẬN (0235708)
# (Hãy điều chỉnh PORT tương ứng với cổng cắm RockBLOCK 0235708 của bạn)
# ==============================================================================
PORT = "COM16"          # Ví dụ: COM16 (hoặc COM4 nếu test mô phỏng com0com)
BAUDRATE = 19200        # Tốc độ baud chuẩn của RockBLOCK 9603 là 19200
TIMEOUT = 3             # Thời gian timeout đọc serial (giây)
LOG_FILE = "receiver_0235708_log.txt"

print(f"\n⚙️ Cấu hình thiết bị nhận: Port = {PORT}, Baud = {BAUDRATE}, Log = {LOG_FILE}")


In [ ]:
# CELL 2: ĐỊNH NGHĨA CLASS ĐIỀU KHIỂN ROCKBLOCK RECEIVER
class RockBlockReceiver:
    """Lớp điều khiển module RockBLOCK nhận tín hiệu MT từ mạng vệ tinh Iridium."""
    
    MT_STATUS_DESC = {
        0: "Không có tin nhắn MT nào chờ trong hộp thư Gateway.",
        1: "Thành công: Đã nhận được 1 tin nhắn MT từ Gateway về bộ đệm modem!",
        2: "Lỗi trong quá trình nhận tin nhắn MT từ Gateway."
    }

    def __init__(self, port, baudrate=19200, timeout=3, log_file="receiver_log.txt"):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.log_file = log_file
        self.ser = None
        self.imei = None

    def log(self, message):
        """In log ra console và lưu vào file log với timestamp."""
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{now}] {message}"
        print(log_entry)
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(log_entry + "\n")

    def connect(self):
        """Mở cổng Serial kết nối tới RockBLOCK."""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout
            )
            self.log(f"✅ Đã kết nối thành công tới cổng {self.port} @ {self.baudrate} baud.")
            return True
        except serial.SerialException as e:
            self.log(f"❌ Lỗi mở cổng Serial: {e}")
            return False

    def disconnect(self):
        """Đóng kết nối Serial an toàn."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.log("🔌 Đã ngắt kết nối cổng Serial.")

    def send_cmd(self, cmd, wait=1.5):
        """Gửi lệnh AT tới modem và đọc phản hồi."""
        if not self.ser or not self.ser.is_open:
            self.log("❌ Cổng Serial chưa mở!")
            return ""
        
        self.ser.reset_input_buffer()
        self.ser.write((cmd + "\r").encode("ascii"))
        time.sleep(wait)
        
        response = self.ser.read_all().decode("ascii", errors="ignore").strip()
        return response

    def init_modem(self):
        """Khởi tạo cấu hình ban đầu cho RockBLOCK."""
        self.log("--- Bắt đầu khởi tạo cấu hình modem bên nhận ---")
        
        # 1. Kiểm tra phản hồi cơ bản
        resp = self.send_cmd("AT", wait=1)
        if "OK" not in resp:
            self.log(f"⚠️ Modem không phản hồi lệnh AT: {resp}")
            return False
            
        # 2. Tắt echo (ATE0)
        self.send_cmd("ATE0", wait=1)
        
        # 3. Tắt phần cứng flow control (AT&K0)
        self.send_cmd("AT&K0", wait=1)
        
        # 4. Lấy thông tin firmware và IMEI
        fw = self.send_cmd("AT+CGMR", wait=1.5)
        self.imei = self.send_cmd("AT+CGSN", wait=1.5).replace("OK", "").strip()
        
        self.log(f"📱 IMEI thiết bị nhận: {self.imei}")
        self.log(f"💾 Firmware          : {fw.replace('OK', '').strip()}")
        self.log("✅ Modem nhận đã sẵn sàng!")
        return True

    def check_signal(self):
        """Kiểm tra mức sóng vệ tinh Iridium (AT+CSQ). Trả về 0-5 vạch."""
        resp = self.send_cmd("AT+CSQ", wait=2)
        if "+CSQ:" in resp:
            try:
                bars_str = resp.split("+CSQ:")[1].strip().split()[0]
                bars = int(bars_str)
                bar_display = "■" * bars + "□" * (5 - bars)
                self.log(f"📶 Mức tín hiệu vệ tinh: [{bar_display}] ({bars}/5 vạch)")
                return bars
            except Exception:
                pass
        self.log(f"⚠️ Không đọc được mức sóng. Phản hồi: {resp}")
        return 0

    def read_buffer_content(self):
        """
        Đọc nội dung tin nhắn văn bản từ MT Buffer bằng lệnh AT+SBDRT.
        Sau đó xóa buffer bằng AT+SBDD1.
        """
        self.log("📖 Đang đọc nội dung từ MT Buffer (AT+SBDRT)...")
        raw_msg = self.send_cmd("AT+SBDRT", wait=1.5)
        
        # Dọn dẹp chuỗi phản hồi
        cleaned_msg = raw_msg
        lines = [line.strip() for line in raw_msg.splitlines() if line.strip() and line.strip() != "OK"]
        if lines:
            cleaned_msg = "\n".join(lines)

        self.log("=" * 60)
        self.log("📩 NỘI DUNG TIN NHẮN NHẬN ĐƯỢC:")
        self.log(f"👉 \"{cleaned_msg}\"")
        self.log("=" * 60)

        # Xóa MT Buffer sau khi đã đọc xong
        self.send_cmd("AT+SBDD1", wait=1)
        return cleaned_msg

    def check_mailbox(self, wait_sbd=20):
        """
        Thực hiện kiểm tra hộp thư Iridium bằng lệnh AT+SBDIX.
        Trả về: (has_message: bool, message_text: str hoặc None, queued_count: int)
        """
        self.log("📡 Đang gửi lệnh AT+SBDIX để kiểm tra hộp thư vệ tinh (Mailbox Check)...")
        sbdix_resp = self.send_cmd("AT+SBDIX", wait=wait_sbd)
        self.log(f"📥 Phản hồi phiên SBDIX:\n{sbdix_resp}")

        if "+SBDIX:" in sbdix_resp:
            try:
                # Cú pháp: +SBDIX: <mo_stat>, <momsn>, <mt_stat>, <mtmsn>, <mt_len>, <mt_queued>
                params = sbdix_resp.split("+SBDIX:")[1].strip().split(",")
                mt_status = int(params[2].strip())
                mtmsn = int(params[3].strip())
                mt_len = int(params[4].strip())
                mt_queued = int(params[5].strip())

                desc = self.MT_STATUS_DESC.get(mt_status, f"Mã trạng thái MT không xác định: {mt_status}")
                self.log(f"📊 Kết quả MT Status = {mt_status}: {desc}")
                self.log(f"🔢 MTMSN: {mtmsn} | Độ dài: {mt_len} bytes | Còn trong hàng đợi: {mt_queued}")

                if mt_status == 1:
                    # Có tin nhắn MT! Tiến hành đọc nội dung
                    msg_text = self.read_buffer_content()
                    return True, msg_text, mt_queued
                else:
                    return False, None, mt_queued

            except Exception as e:
                self.log(f"❌ Lỗi giải mã phản hồi SBDIX: {e}")
                return False, None, 0
        else:
            self.log("⚠️ Không nhận được phản hồi +SBDIX từ modem.")
            return False, None, 0

    def listen_loop(self, poll_interval=30, max_cycles=10):
        """
        Chế độ lắng nghe liên tục: Định kỳ kiểm tra hộp thư để nhận tin nhắn mới.
        - poll_interval: Thời gian nghỉ giữa các lần kiểm tra (giây).
        - max_cycles: Số chu kỳ lặp (None nếu muốn chạy vô tận tới khi bấm Stop).
        """
        self.log("=" * 60)
        self.log(f"🎧 BẬT CHẾ ĐỘ TRỰC NHẬN TIN NHẮN (Chu kỳ thăm dò: {poll_interval}s)")
        self.log("👉 Nhấn nút [Stop / Interrupt Kernel] trong Jupyter để dừng lắng nghe bất cứ lúc nào.")
        self.log("=" * 60)

        cycle = 0
        try:
            while True:
                cycle += 1
                self.log(f"\n--- Chu kỳ kiểm tra {cycle}" + (f"/{max_cycles}" if max_cycles else "") + " ---")
                
                # Kiểm tra sóng trước khi gọi SBDIX
                bars = self.check_signal()
                if bars >= 1:
                    has_msg, msg, queued = self.check_mailbox(wait_sbd=20)
                    if has_msg:
                        self.log(f"🎉 ĐÃ NHẬN ĐƯỢC TIN NHẮN MỚI THÀNH CÔNG!")
                        
                        # Nếu còn tin nhắn khác đang chờ trong hàng đợi Gateway, kéo về tiếp
                        while queued > 0:
                            self.log(f"📦 Vẫn còn {queued} tin nhắn trong hàng đợi Gateway. Đang tải tiếp...")
                            time.sleep(3)
                            has_next, next_msg, queued = self.check_mailbox(wait_sbd=20)
                    else:
                        self.log("📭 Hộp thư trống, chưa có tin nhắn mới.")
                else:
                    self.log("⚠️ Sóng vệ tinh 0 vạch. Tạm bỏ qua chu kỳ này để tiết kiệm thời gian.")

                if max_cycles and cycle >= max_cycles:
                    self.log("⏹️ Đã đạt giới hạn chu kỳ kiểm tra.")
                    break

                self.log(f"💤 Nghỉ {poll_interval} giây trước lần kiểm tra kế tiếp...")
                time.sleep(poll_interval)

        except KeyboardInterrupt:
            self.log("🛑 Người dùng đã bấm dừng chế độ lắng nghe.")


In [ ]:
# CELL 3: KHỞI TẠO ĐỐI TƯỢNG VÀ KẾT NỐI VỚI ROCKBLOCK RECEIVER
receiver = RockBlockReceiver(port=PORT, baudrate=BAUDRATE, log_file=LOG_FILE)

# Mở cổng kết nối
if receiver.connect():
    # Cấu hình modem và đọc IMEI
    ready = receiver.init_modem()
    if not ready:
        print("⚠️ Khởi tạo modem thất bại. Vui lòng kiểm tra cổng COM hoặc rút cắm lại thiết bị.")
else:
    print("❌ Không mở được cổng Serial. Hãy kiểm tra lại biến PORT ở Cell 1!")


In [ ]:
# CELL 4: KIỂM TRA SÓNG VỆ TINH
# Đảm bảo anten của RockBLOCK 0235708 được đặt ngoài trời thoáng
print("📡 Kiểm tra mức sóng của thiết bị nhận:")
receiver.check_signal()


In [ ]:
# CELL 5: KIỂM TRA HỘP THƯ 1 LẦN (MANUAL CHECK MAILBOX)
# Thực hiện kiểm tra xem thiết bị 0235707 đã gửi tin nhắn tới chưa
print("📬 Đang thực hiện kiểm tra hộp thư (One-shot check)...")
has_message, message_content, remaining_queued = receiver.check_mailbox(wait_sbd=20)

if has_message:
    print(f"\n🎉 NHẬN THÀNH CÔNG: {message_content}")
else:
    print("\n📭 Hiện chưa có tin nhắn nào trong hộp thư.")


In [ ]:
# CELL 6: BẬT CHẾ ĐỘ LẮNG NGHE ĐỊNH KỲ (CONTINUOUS LISTENING)
# Chạy vòng lặp tự động thăm dò hộp thư mỗi 30 giây.
# Bạn có thể bấm nút Stop (Interrupt kernel) trên thanh công cụ Jupyter khi muốn dừng.
receiver.listen_loop(poll_interval=30, max_cycles=5)


In [ ]:
# CELL 7: ĐÓNG KẾT NỐI KHI HOÀN TẤT
receiver.disconnect()
print(f"📄 Toàn bộ log nhận tin đã được ghi vào file: {LOG_FILE}")
